# Fine-Tune RoBERTa for Discourse Role Classification

Fine-tunes `roberta-base` on the **PERSUADE 2.0** corpus as a 7-class sentence classifier using all native PERSUADE labels — a drop-in replacement for the zero-shot pipeline in `discourse_analysis.ipynb`.

## 1. GPU & Reproducibility Setup

In [ ]:
# Uncomment to install / upgrade if needed
# !pip install transformers datasets accelerate scikit-learn seaborn --upgrade


In [ ]:
import os, random
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 0 if torch.cuda.is_available() else -1
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

BASE_MODEL = 'roberta-base'
OUTPUT_DIR = './roberta-discourse-finetuned-v2'


## 2. Load & Inspect the PERSUADE 2.0 Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd

# Option A: load from HuggingFace Hub (may require `huggingface-cli login` if gated)
# Try both known identifiers for PERSUADE 2.0
# try:
#     raw = load_dataset('scrosseye/persuade_corpus_2.0')
# except Exception:
#     raw = load_dataset('persuade-corpus/persuade_2.0')

# Option B: load from a local CSV downloaded from Kaggle / the PERSUADE repo
# Uncomment and set the path if Hub access fails:
raw = load_dataset('csv', data_files={'train': './data/train.csv'})

print(raw)
print('\nSample row:', raw['train'][0])


In [ ]:
train_df = raw['train'].to_pandas()
print(f'Total spans: {len(train_df):,}')
print(train_df['discourse_type'].value_counts())


## 3. Label Setup & Class Distribution


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight

# Use all 7 native PERSUADE 2.0 labels — no merging
LABELS   = ['Lead', 'Position', 'Claim', 'Evidence', 'Counterclaim', 'Rebuttal', 'Concluding Statement']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}

train_df['label_str'] = train_df['discourse_type']
train_df['label']     = train_df['label_str'].map(LABEL2ID)
train_df = train_df.dropna(subset=['label'])
train_df['label'] = train_df['label'].astype(int)

fig, ax = plt.subplots(figsize=(9, 4))
counts = train_df['label_str'].value_counts().reindex(LABELS)
sns.barplot(x=counts.index, y=counts.values, ax=ax)
ax.set_title('Discourse span class distribution (PERSUADE 2.0, all 7 labels)')
ax.set_ylabel('Count'); plt.xticks(rotation=20, ha='right'); plt.tight_layout(); plt.show()

# Class weights shown here for reference only.
# Training uses focal loss instead of weighted cross-entropy — focal loss
# handles imbalance dynamically based on per-example model confidence,
# so a fixed per-class multiplier is not needed.
class_weights = compute_class_weight('balanced', classes=np.arange(len(LABELS)), y=train_df['label'].values)
print('Class imbalance reference (not passed to Trainer — focal loss handles this):')
print(dict(zip(LABELS, class_weights.round(3))))


## 4. Tokenisation with RoBERTa Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
print(f'Tokenizer type : {type(tokenizer).__name__}')
print(f'Available columns: {list(train_df.columns)}')

MAX_LEN = 256

# ── Resolve column names ──────────────────────────────────────────────────────
# Different PERSUADE / Feedback Prize CSV releases use different column names.

if 'essay_id' not in train_df.columns:
    raise ValueError(f"No 'essay_id' column found. Available: {list(train_df.columns)}")

# Text column: the discourse span content
TEXT_COL_CANDIDATES = ['discourse_text', 'text', 'essay_text', 'sentence', 'span_text']
TEXT_COL = next((c for c in TEXT_COL_CANDIDATES if c in train_df.columns), None)
if TEXT_COL is None:
    raise ValueError(f"Cannot find discourse text column. Available: {list(train_df.columns)}")
print(f"Using text column: '{TEXT_COL}'")

# ── Determine span ordering within each essay ─────────────────────────────────
if 'discourse_start' in train_df.columns:
    sort_col = 'discourse_start'
    print("Sorting by 'discourse_start'.")
elif 'predictionstring' in train_df.columns:
    train_df['discourse_start'] = (
        train_df['predictionstring'].str.split().str[0].astype(int)
    )
    sort_col = 'discourse_start'
    print("Derived 'discourse_start' from 'predictionstring'.")
else:
    train_df['discourse_start'] = train_df.groupby('essay_id').cumcount()
    sort_col = 'discourse_start'
    print("No position column found — using row order within each essay.")

# ── Sort spans into document order ────────────────────────────────────────────
train_df_sorted = (
    train_df
    .sort_values(['essay_id', sort_col])
    .reset_index(drop=True)
)

# ── Build context windows ─────────────────────────────────────────────────────
# Discourse roles are relational: a Rebuttal is only labelled as such because a
# Counterclaim precedes it. Without context, the model sees each span in isolation.
#
# Format: [prev span] </s> [target span] </s> [next span]
# Spans at essay boundaries get a one-sided window (no prev or no next).
SEP = tokenizer.sep_token   # '</s>' for RoBERTa

train_df_sorted['_prev'] = train_df_sorted[TEXT_COL].shift(1, fill_value='')
train_df_sorted['_next'] = train_df_sorted[TEXT_COL].shift(-1, fill_value='')

same_as_prev = train_df_sorted['essay_id'] == train_df_sorted['essay_id'].shift(1).fillna('')
same_as_next = train_df_sorted['essay_id'] == train_df_sorted['essay_id'].shift(-1).fillna('')

train_df_sorted.loc[~same_as_prev, '_prev'] = ''
train_df_sorted.loc[~same_as_next, '_next'] = ''

def _make_context(row):
    parts = [p for p in [row['_prev'], row[TEXT_COL], row['_next']] if p]
    return f' {SEP} '.join(parts)

train_df_sorted['context_text'] = train_df_sorted.apply(_make_context, axis=1)
train_df_sorted.drop(columns=['_prev', '_next'], inplace=True)

n_essays = train_df_sorted['essay_id'].nunique()
print(f'Built context windows for {len(train_df_sorted):,} spans across {n_essays:,} essays.')
print('\nExample context (span 1 of first essay):')
idx = train_df_sorted[train_df_sorted['essay_id'] == train_df_sorted['essay_id'].iloc[0]].index
if len(idx) > 1:
    print(train_df_sorted.at[idx[1], 'context_text'][:500])


## 5. Essay-Level Train / Val / Test Splits (80 / 10 / 10)


In [ ]:
from datasets import Dataset, DatasetDict, ClassLabel, Features, Value

# ── Essay-level split (80 / 10 / 10) ─────────────────────────────────────────
# Random span-level splitting (the previous approach) leaks essay-specific style
# and vocabulary into the test set, inflating test performance. An essay with a
# distinctive opening sentence in train would make its Evidence spans trivially
# recognisable at test time. Splitting on essay_id ensures no essay appears in
# more than one partition — a harder but honest evaluation.

essay_ids = train_df_sorted['essay_id'].unique().copy()
rng = np.random.default_rng(SEED)
rng.shuffle(essay_ids)

n       = len(essay_ids)
n_train = int(0.8 * n)
n_val   = int(0.1 * n)

train_ids = set(essay_ids[:n_train])
val_ids   = set(essay_ids[n_train : n_train + n_val])
test_ids  = set(essay_ids[n_train + n_val :])

df_train = train_df_sorted[train_df_sorted['essay_id'].isin(train_ids)]
df_val   = train_df_sorted[train_df_sorted['essay_id'].isin(val_ids)]
df_test  = train_df_sorted[train_df_sorted['essay_id'].isin(test_ids)]

print(f'Essays  — train: {len(train_ids):,}  val: {len(val_ids):,}  test: {len(test_ids):,}')
print(f'Spans   — train: {len(df_train):,}  val: {len(df_val):,}  test: {len(df_test):,}')

# ── Tokenise each split separately ───────────────────────────────────────────
def make_hf_dataset(df_subset):
    """Tokenize a span subset and return a formatted HF Dataset."""
    features = Features({
        'context_text': Value('string'),
        'labels':       ClassLabel(names=LABELS),
    })
    ds = Dataset.from_dict({
        'context_text': df_subset['context_text'].tolist(),
        'labels':       df_subset['label'].tolist(),
    }, features=features)
    def tokenize_fn(batch):
        return tokenizer(batch['context_text'], truncation=True,
                         max_length=MAX_LEN, padding='max_length')
    ds = ds.map(tokenize_fn, batched=True, batch_size=512)
    ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return ds

dataset = DatasetDict({
    'train': make_hf_dataset(df_train),
    'val':   make_hf_dataset(df_val),
    'test':  make_hf_dataset(df_test),
})
print(dataset)

# Verify CLS token present
first_ids = dataset['train'][0]['input_ids'][:5].tolist()
cls_id    = tokenizer.cls_token_id
assert first_ids[0] == cls_id, f'MISSING CLS TOKEN — got {first_ids[0]}, expected {cls_id}'
print('Special tokens: OK')


## 6. Model — RoBERTa for Sequence Classification

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
print(f'Parameters : {sum(p.numel() for p in model.parameters()):,}')
print(f'Dtype      : {next(model.parameters()).dtype}')  # should be float32


## 7. Training Arguments & Metrics

In [ ]:
from transformers import TrainingArguments
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro'),
            'weighted_f1': f1_score(labels, preds, average='weighted')}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=16,   # reduced from 32 to fit MAX_LEN=256 in 8GB VRAM
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='linear',
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=50,
    logging_nan_inf_filter=False,
    report_to='none',
    seed=SEED,
)
print(f'fp16={training_args.fp16}  bf16={training_args.bf16}')


## 8. Trainer with Focal Loss


In [ ]:
from transformers import Trainer
import torch.nn.functional as F


class FocalLossTrainer(Trainer):
    """
    Trainer with focal loss to address class imbalance.

    Unlike class weights (which apply a fixed per-class multiplier regardless of
    model confidence), focal loss dynamically down-weights easy examples based on
    how confident the model already is about them:

        FL(p_t) = -(1 - p_t)^gamma * log(p_t)

    When gamma=0 this reduces to standard cross-entropy. At gamma=2 (the default
    from Lin et al.), a well-classified example with p_t=0.9 receives a weight of
    (1-0.9)^2 = 0.01 — 100x less than a misclassified example with p_t=0.1.

    This is particularly useful for rare but hard classes (Rebuttal, Counterclaim):
    they are both infrequent AND semantically similar to adjacent roles, so the
    model is uncertain about them even on training examples it sees often.

    Reference: Lin et al. (2017) "Focal Loss for Dense Object Detection", ICCV.
               https://arxiv.org/abs/1708.02002
    """

    def __init__(self, *args, gamma: float = 2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits

        # Log-probabilities and probabilities from the logits
        log_probs = F.log_softmax(logits, dim=-1)
        probs     = log_probs.exp()

        # p_t: model's probability on the true class for each example
        p_t = probs.gather(dim=1, index=labels.unsqueeze(1)).squeeze(1)

        # Focal modulation: down-weight easy examples exponentially
        focal_weight = (1.0 - p_t) ** self.gamma

        # Standard NLL loss per example, then apply focal weight
        ce_loss = F.nll_loss(log_probs, labels, reduction='none')
        loss    = (focal_weight * ce_loss).mean()

        return (loss, outputs) if return_outputs else loss


trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['val'],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    gamma=2.0,
)


## 9. Train

In [ ]:
train_result = trainer.train()
print(train_result)


In [ ]:
log_history = trainer.state.log_history
train_loss = [(e['epoch'], e['loss'])      for e in log_history if 'loss' in e and 'eval_loss' not in e]
eval_rows  = [(e['epoch'], e['eval_loss'], e['eval_macro_f1']) for e in log_history if 'eval_loss' in e]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
if train_loss:
    ep, loss = zip(*train_loss); axes[0].plot(ep, loss, label='train loss', alpha=0.6)
if eval_rows:
    ep, eloss, f1 = zip(*eval_rows)
    axes[0].plot(ep, eloss, marker='o', label='val loss')
    axes[1].plot(ep, f1, marker='o', color='green')
axes[0].set_title('Loss curves'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].set_title('Macro-F1 (validation)'); axes[1].set_xlabel('Epoch')
plt.tight_layout(); plt.show()


## 10. Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

preds_out = trainer.predict(dataset['test'])
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = preds_out.label_ids
print(classification_report(y_true, y_pred, target_names=LABELS, digits=3))


In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_title('Normalised confusion matrix (test set)')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.xticks(rotation=25, ha='right'); plt.tight_layout(); plt.show()


## 11. Error Analysis — Most Confused Examples

In [ ]:
import torch.nn.functional as F

softmax_probs = F.softmax(torch.tensor(preds_out.predictions), dim=-1).numpy()
test_df = dataset['test'].to_pandas()
test_df['true_label'] = [ID2LABEL[i] for i in y_true]
test_df['pred_label'] = [ID2LABEL[i] for i in y_pred]
test_df['confidence'] = softmax_probs.max(axis=1)
test_df['correct']    = test_df['true_label'] == test_df['pred_label']

errors = test_df[~test_df['correct']].sort_values('confidence', ascending=False)
print(f'Errors: {len(errors):,} / {len(test_df):,} ({100*len(errors)/len(test_df):.1f}%)')
errors[['context_text', 'true_label', 'pred_label', 'confidence']].head(10)


In [ ]:
confusion_pairs = (
    errors.groupby(['true_label', 'pred_label']).size()
    .reset_index(name='count').sort_values('count', ascending=False)
)
print(confusion_pairs.head(15).to_string(index=False))


## 12. Save & Export Fine-Tuned Model

In [ ]:
from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')
# Optional: trainer.push_to_hub('your-username/deberta-discourse-persuade')


## 13. Drop-in Replacement Smoke Test

Load the fine-tuned model as a `text-classification` pipeline and confirm it runs correctly.

**Already updated in `discourse_analysis.ipynb`:**
- `FINETUNED_MODEL` now points to `./roberta-discourse-finetuned-v2`
- `label_sentences()` builds tri-span context windows (`[prev] </s> [target] </s> [next]`) to match training format

> **Important:** This model was trained on context windows. Running it on isolated sentences at inference time will recover much less of the +0.1 F1 gain. Always pass context strings at inference.


In [ ]:
import spacy
from transformers import pipeline as hf_pipeline

nlp = spacy.load('en_core_web_sm', disable=['ner', 'lemmatizer'])

def split_sentences(text):
    return [s.text.strip() for s in nlp(str(text)).sents if s.text.strip()]

ft_classifier = hf_pipeline('text-classification', model=OUTPUT_DIR, tokenizer=OUTPUT_DIR, device=DEVICE)

def label_sentences_ft(sentences, batch_size=64):
    if not sentences: return []
    results = ft_classifier(sentences, batch_size=batch_size, truncation=True, max_length=MAX_LEN)
    return [r['label'] for r in results]

zs_classifier = hf_pipeline('zero-shot-classification', model=BASE_MODEL, device=DEVICE)

def label_sentences_zs(sentences, batch_size=32):
    if not sentences: return []
    results = zs_classifier(sentences, candidate_labels=LABELS, multi_label=False, batch_size=batch_size)
    if isinstance(results, dict): results = [results]
    return [r['labels'][0] for r in results]

DUMMY_ESSAYS = [
    'Growing up in a small town, I never imagined standing before a crowd. '
    'I struggled with anxiety for years. But joining debate changed me. '
    'I learned my voice matters. Now I want to study psychology to help others find theirs.',
    'My mother works two jobs. Every morning I watch her leave before sunrise. '
    'That image drives everything I do. I maintain a 4.0 GPA not for myself but for her. '
    'I intend to become an engineer and give her the life she deserves.',
    "I have always believed science can solve humanity's greatest problems. "
    'Studies show renewable energy can reduce emissions by up to 70%. '
    'I conducted my own solar panel efficiency research and the data confirmed '
    'innovation, not incremental change, is what is required.',
]

rows = []
for essay in DUMMY_ESSAYS:
    sents = split_sentences(essay)
    ft_labels = label_sentences_ft(sents)
    zs_labels = label_sentences_zs(sents)
    for sent, ft, zs in zip(sents, ft_labels, zs_labels):
        rows.append({'sentence': sent[:80], 'fine_tuned': ft, 'zero_shot': zs, 'agree': ft == zs})

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
print(f"\nAgreement rate: {comparison['agree'].mean():.1%}")
